# 03 — Comparaison des Embeddings

## Objectif

Comparer plusieurs modèles d'embedding avec le même LLM et les mêmes
paramètres de retrieval.

**Question de recherche :** Quel modèle d'embedding offre la meilleure
qualitée de retrieval pour les documents académiques ?

**Paramètres fixes :**
- LLM : Gemini 2.5 Flash
- Retrieval : top_k=3
- 5 questions de test

---
**Pourquoi cette comparaison est importante :**
L'embedding détermine la qualité du retrieval. Un mauvais embedding
peut faire échouer tout le pipeline RAG, même avec un bon LLM.

## 1. Imports

In [1]:
import sys
from pathlib import Path
import time

ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "src").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

from config import LLMConfig, RetrievalConfig, EmbeddingConfig
from retriever import retrieve_documents
from llm_chain import generate_answer
from evaluation_judge import create_judge
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

print("✅ Imports réussis")

✅ Imports réussis


## 2. Paramètres fixes

In [2]:
llm_cfg = LLMConfig(
    provider="openrouter", model="google/gemini-2.5-flash",
    temperature=0.2, num_predict=512, request_timeout=120,
)
ret_cfg = RetrievalConfig(top_k=3, max_distance=1.5)

print(f"🤖 LLM       : {llm_cfg.provider}:{llm_cfg.model}")
print(f"📄 Retrieval : top_k={ret_cfg.top_k}")

🤖 LLM       : openrouter:google/gemini-2.5-flash
📄 Retrieval : top_k=3


## 3. Modèles d'embedding à comparer

**Note :** seuls les modèles déjà ingérés (index FAISS présent dans
`vectorstore/<slug>/`) peuvent être testés.

In [3]:
EMBEDDINGS = [
    ("BGE-M3",     "huggingface", "BAAI/bge-m3"),
    ("E5-Large",   "huggingface", "intfloat/multilingual-e5-large"),
    ("Jina v3",    "huggingface", "jinaai/jina-embeddings-v3"),
    ("Qwen Embed",  "ollama",      "qwen3-embedding"),
]

# Vérifier quels indexes existent
available = []
for name, provider, model in EMBEDDINGS:
    slug = model.replace("/", "_")
    if (ROOT / "vectorstore" / slug).exists():
        available.append((name, provider, model))
        print(f"✅ {name:12s} → index disponible")
    else:
        print(f"❌ {name:12s} → index manquant (lancez ingest.py)")

print(f"\n📋 {len(available)} embedding(s) disponible(s)")

✅ BGE-M3       → index disponible
❌ E5-Large     → index manquant (lancez ingest.py)
❌ Jina v3      → index manquant (lancez ingest.py)
✅ Qwen Embed   → index disponible

📋 2 embedding(s) disponible(s)


## 4. Questions de test

In [4]:
QUESTIONS = [
    ("Q01", "Quelle est la note minimale pour valider un module ?",
     "L'étudiant doit obtenir une note finale >= 5,5/10"),
    ("Q02", "Quel est le montant des frais de prolongation ?",
     "Les frais de prolongation sont de 1 000 000 VND par mois"),
    ("Q03", "Peut-on demander une prolongation du mémoire ?",
     "Oui, avec une demande écrite au Service de scolarité"),
]

print(f"📚 {len(QUESTIONS)} questions")

📚 3 questions


## 5. Juge DeepEval

In [5]:
judge = create_judge(provider="ollama", model="qwen2.5:3b")
print(f"⚖️  Juge : {judge.get_model_name()}")

⚖️  Juge : qwen2.5:3b


## 6. Boucle d'évaluation

Pour chaque modèle d'embedding, on exécute le pipeline complet.

In [6]:
results = []

for emb_name, provider, model in available:
    print(f"\n{'='*60}")
    print(f"  🔄 Test : {emb_name}")
    print(f"{'='*60}")

    emb_cfg = EmbeddingConfig(provider=provider, model=model, device="cpu")

    for qid, question, expected in QUESTIONS::
        try:
        docs, scores = retrieve_documents(question, ret_cfg, emb_cfg)

        start = time.time()
        answer = generate_answer(question, docs, llm_cfg)
        elapsed = time.time() - start

        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            expected_output=expected,
            retrieval_context=[d.page_content for d in docs],
        )

        faith = FaithfulnessMetric(threshold=0.75, model=judge, include_reason=True)
        relev = AnswerRelevancyMetric(threshold=0.75, model=judge, include_reason=True)

        try:
            faith.measure(test_case)
        except Exception:
            faith.score = 0.0
        try:
            relev.measure(test_case)
        except Exception:
            relev.score = 0.0

        results.append({
            "Embedding": emb_name,
            "Question": qid,
            "Faithfulness": round(faith.score, 4),
            "AnswerRelevancy": round(relev.score, 4),
            "Temps(s)": round(elapsed, 2),
            "Chunks": len(docs),
        })

        except Exception as e:
            print(f"   [03] ❌ Erreur : {e}")
                    nt(f"   [{qid}] Faith={faith.score:.3f}  Relev={relev.score:.3f}  ({elapsed:.1f}s)")

df = pd.DataFrame(results)
print(f"\n✅ Terminé : {len(df)} mesures")

SyntaxError: invalid syntax (232565078.py, line 10)

## 7. Tableau comparatif

In [ ]:
summary = df.groupby("Embedding")[["Faithfulness", "AnswerRelevancy", "Temps(s)"]].mean().round(4)
summary.columns = ["Fidélité", "Pertinence", "Temps (s)"]
summary

## 8. Graphique

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = df.melt(id_vars=["Embedding"], value_vars=["Faithfulness", "AnswerRelevancy"],
                  var_name="Métrique", value_name="Score")
sns.barplot(data=plot_df, x="Embedding", y="Score", hue="Métrique", ax=ax)
ax.set_title("Comparaison des Embeddings", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 9. Export CSV

In [ ]:
out_dir = ROOT / "evaluation" / "results"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "03_compare_embeddings.csv", index=False)
summary.to_csv(out_dir / "03_compare_embeddings_summary.csv")
print(f"✅ Exporté dans {out_dir}/")

## 10. Analyse

**Lecture des résultats :**
- Un bon embedding améliore le **Faithfulness** (meilleur contexte fourni)
- Certains embeddings peuvent être lents à charger (modèles volumineux)
- Comparez aussi le nombre de chunks récupérés : plus n'est pas toujours mieux

**Recommandation :** choisir l'embedding avec la meilleure fidélité
tout en restant dans un temps de chargement acceptable.